# NC Field Soil Mapping

**Date:** 2026-04-06
**Data Source:** `NC_soil_crop_data_summary_EPSG4326_2026-04-01.csv`

## Summary

This notebook presents interactive maps of 23 agricultural fields in North Carolina with two sets of soil properties:

**Map 1: Soil Characteristics**
- Soil pH
- Organic Matter (%)
- CEC (meq/100g)
- Available Water Storage (inches)

**Map 2: Soil Texture Components**
- Clay (%)
- Sand (%)
- Silt (%)

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

DATA_DIR = Path("/workspaces/my-farm-advisor/data/workspace")
OUTPUT_DIR = DATA_DIR / "output"

## 1. Interactive Map: Soil Characteristics

This map shows pH, Organic Matter, CEC, and Available Water Storage for each field. Use the dropdown menu in the bottom-left corner or the layer control in the top-right to switch between layers.

In [ ]:
from IPython.display import IFrame

IFrame(src='../output/farm_field_map_interactive.html', width=900, height=600)

## 2. Interactive Map: Soil Texture Components

This map shows Clay, Sand, and Silt percentages for each field. These three components together make up 100% of soil texture.

In [ ]:
IFrame(src='../output/farm_field_soil_components_interactive.html', width=900, height=600)

## 3. Summary Statistics

The following table summarizes all soil properties across the 23 fields.

In [ ]:
df = pd.read_csv(DATA_DIR / "NC_soil_crop_data_summary_EPSG4326_2026-04-01.csv")

# Select key soil properties
soil_props = ['avg_ph', 'avg_om_pct', 'avg_cec', 'total_aws_inches', 
              'avg_clay_pct', 'avg_sand_pct', 'avg_silt_pct', 'avg_bulk_density']

# Calculate summary statistics
summary = df[soil_props].describe().T
summary = summary[['min', 'max', 'mean', 'std']].round(2)
summary.columns = ['Min', 'Max', 'Mean', 'Std Dev']
summary.index = ['pH', 'Organic Matter (%)', 'CEC (meq/100g)', 'Water Storage (in)', 
                 'Clay (%)', 'Sand (%)', 'Silt (%)', 'Bulk Density (g/cm³)']

summary

## 4. Soil Texture Distribution

Soil texture (clay, sand, silt) is a fundamental property that affects water retention, nutrient availability, and tillage management.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create ternary-style summary
texture_means = df[['avg_clay_pct', 'avg_sand_pct', 'avg_silt_pct']].mean()

fig, ax = plt.subplots(figsize=(8, 4))

x = np.arange(3)
width = 0.6

bars = ax.bar(x, texture_means.values, width, color=['#8B4513', '#F4A460', '#D3D3D3'])
ax.set_ylabel('Percentage (%)')
ax.set_title('Average Soil Texture Components Across All Fields')
ax.set_xticks(x)
ax.set_xticklabels(['Clay', 'Sand', 'Silt'])

for bar, val in zip(bars, texture_means.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
            f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')

ax.set_ylim(0, 60)
plt.tight_layout()
plt.show()

## 5. Field Summary

The following table shows each field with its key soil properties.

In [ ]:
display_cols = ['field_id', 'area_acres', 'avg_ph', 'avg_om_pct', 'avg_cec', 
                'avg_clay_pct', 'avg_sand_pct', 'avg_silt_pct', 'total_aws_inches']

field_summary = df[display_cols].copy()
field_summary.columns = ['Field ID', 'Acres', 'pH', 'OM (%)', 'CEC', 'Clay (%)', 'Sand (%)', 'Silt (%)', 'AWS (in)']

pd.set_option('display.max_rows', 25)
field_summary.round(2)

## 6. Soil Analysis Narrative

### Acidity

The fields in this dataset are **generally acidic**, with pH values ranging from 4.81 to 6.08 and a mean of 5.42. This is below the optimal pH range (6.0-7.0) for most row crops. Of the 23 fields, **20 fields (87%) have pH below 6.0**, indicating a need for lime application to neutralize acidity for optimal crop production.

### Geographic Trends

Analysis of field locations reveals several geographic patterns:

**North-to-South Trends (Latitude):**
- **pH increases northward** (correlation +0.49): Northern fields tend to be less acidic. The most northern field (osm-813157720 at lat 36.34°) has a pH of 5.46, while southern fields like osm-1153259427 (lat 34.87°) have pH as low as 4.81.
- **Organic Matter decreases northward** (correlation -0.35): Southern fields tend to have higher organic matter. Field osm-1133139440 (lat 34.93°) has the highest OM at 2.18%.
- **Clay content increases northward** (correlation +0.56): Northern fields have higher clay content, which correlates with better water retention and nutrient-holding capacity.
- **Available Water Storage increases northward** (correlation +0.53): This aligns with the higher clay content in northern fields.

**East-to-West Trends (Longitude):**
- **Sand content increases westward** (correlation +0.38): Western fields tend to be sandier, while eastern fields have higher clay content.
- **Clay content decreases westward** (correlation -0.33): Eastern fields tend to have more clay.
- **pH decreases westward** (correlation -0.30): Western fields tend to be more acidic.
- **CEC decreases westward** (correlation -0.27): Eastern fields have higher CEC, consistent with their higher clay content.

### Management Implications

Based on these patterns:
- **Southern and western fields** may benefit most from lime applications to address acidity
- **Southern fields** appear to have a natural advantage with higher organic matter, which can improve soil structure and nutrient availability
- **Northern and eastern fields** have better water storage capacity due to higher clay content, which may reduce irrigation needs
- **Western fields** with sandy soils may require more frequent irrigation and fertilizer applications due to lower water and nutrient retention

In [ ]:
# Visualize geographic trends
gdf = gpd.read_file(DATA_DIR / "NC_field_boundaries_EPSG4326_2026-04-01.geojson")
gdf = gdf.merge(df, on='field_id')
gdf['centroid'] = gdf.geometry.centroid
gdf['lon'] = gdf.centroid.x
gdf['lat'] = gdf.centroid.y

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# pH vs Latitude
ax1 = axes[0]
scatter1 = ax1.scatter(gdf['lat'], gdf['avg_ph'], c=gdf['avg_ph'], cmap='YlOrRd', s=80, edgecolor='black')
ax1.set_xlabel('Latitude (°N)')
ax1.set_ylabel('Soil pH')
ax1.set_title('pH vs Latitude\n(Correlation: +0.49)')
z1 = np.polyfit(gdf['lat'], gdf['avg_ph'], 1)
p1 = np.poly1d(z1)
ax1.plot(gdf['lat'].sort_values(), p1(gdf['lat'].sort_values()), "r--", alpha=0.7, label='Trend')
ax1.legend()

# OM vs Latitude
ax2 = axes[1]
scatter2 = ax2.scatter(gdf['lat'], gdf['avg_om_pct'], c=gdf['avg_om_pct'], cmap='YlGn', s=80, edgecolor='black')
ax2.set_xlabel('Latitude (°N)')
ax2.set_ylabel('Organic Matter (%)')
ax2.set_title('Organic Matter vs Latitude\n(Correlation: -0.35)')
z2 = np.polyfit(gdf['lat'], gdf['avg_om_pct'], 1)
p2 = np.poly1d(z2)
ax2.plot(gdf['lat'].sort_values(), p2(gdf['lat'].sort_values()), "r--", alpha=0.7, label='Trend')
ax2.legend()

plt.tight_layout()
plt.show()

print("Note: Southern fields (lower latitude) tend to have higher organic matter, while northern fields tend to have higher pH.")

---

**Interactive Maps:**
- `farm_field_map_interactive.html` - Soil characteristics (pH, OM, CEC, AWS)
- `farm_field_soil_components_interactive.html` - Soil texture (Clay, Sand, Silt)